# 03 — Geography Standardization

## Dengue Disease Prediction for Sri Lanka

### Purpose

This notebook examines geographic identifiers across the dengue, weather and population datasets and prepares a consistent geographic naming system for later dataset integration.

The objectives are to:

- identify geographic names used by each dataset
- detect spelling and formatting inconsistencies
- compare geographic coverage across datasets
- distinguish reporting areas from administrative districts
- define a transparent geographic standardization strategy
- validate the standardized geographic identifiers

The original raw datasets are not modified.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
print("Project root:", PROJECT_ROOT)

Project root: d:\Dengue_Project


In [3]:
DENGUE_DIR = PROJECT_ROOT / "data" / "raw" / "dengue"
WEATHER_DIR = PROJECT_ROOT / "data" / "raw" / "weather"
POPULATION_DIR = PROJECT_ROOT / "data" / "raw" / "population"
print("Dengue directory:", DENGUE_DIR)
print("Weather directory:", WEATHER_DIR)
print("Population directory:", POPULATION_DIR)

Dengue directory: d:\Dengue_Project\data\raw\dengue
Weather directory: d:\Dengue_Project\data\raw\weather
Population directory: d:\Dengue_Project\data\raw\population


In [4]:
dengue_files = list(DENGUE_DIR.glob("*.csv"))
weather_files = list(WEATHER_DIR.glob("*.csv"))
population_files = list(POPULATION_DIR.glob("*.csv"))
print("Dengue CSV files:", len(dengue_files))
print("Weather CSV files:", len(weather_files))
print("Population CSV files:", len(population_files))

Dengue CSV files: 1
Weather CSV files: 1
Population CSV files: 1


In [6]:
dengue_file = dengue_files[0]
weather_file = weather_files[0]
population_file = population_files[0]
print("Dengue:", dengue_file.name)
print("Weather:", weather_file.name)
print("Population:", population_file.name)

Dengue: srilanka_dengue_weekly_2010_2024.csv
Weather: sri_lanka_weekly_weather_2010_2024.csv
Population: district_population_2014_2024.csv


In [8]:
dengue = pd.read_csv(dengue_file)
weather = pd.read_csv(weather_file)
population = pd.read_csv(population_file)

In [9]:
print("Dengue shape:", dengue.shape)
print("Weather shape:", weather.shape)
print("Population shape:", population.shape)

Dengue shape: (20306, 6)
Weather shape: (20332, 19)
Population shape: (475, 3)


In [10]:
print("Dengue has district column:", "district" in dengue.columns)
print("Weather has district column:", "district" in weather.columns)
print("Population has district column:", "district" in population.columns)

Dengue has district column: True
Weather has district column: True
Population has district column: True


## 1. Geographic Identifier Inspection

The unique geographic identifiers in each raw dataset are inspected before any standardization is performed.

In [11]:
dengue_areas = sorted(dengue["district"].dropna().unique())
weather_areas = sorted(weather["district"].dropna().unique())
population_areas = sorted(population["district"].dropna().unique())
print("Dengue areas:", len(dengue_areas))
print("Weather areas:", len(weather_areas))
print("Population areas:", len(population_areas))

Dengue areas: 26
Weather areas: 26
Population areas: 25


In [13]:
print("DENGUE AREAS")
for area in dengue_areas:
    print(area)

DENGUE AREAS
Ampara
Anuradhapura
Badulla
Batticaloa
Colombo
Galle
Gampaha
Hambanthota
Jaffna
Kalmune
Kalutara
Kandy
Kegalle
Kilinochchi
Kurunegala
Mannar
Matale
Matara
Monaragala
Mullaitivu
NuwaraEliya
Polonnaruwa
Puttalam
Ratnapura
Trincomalee
Vavuniya


In [14]:
print("WEATHER AREAS")
for area in weather_areas:
    print(area)

WEATHER AREAS
Ampara
Anuradhapura
Badulla
Batticaloa
Colombo
Galle
Gampaha
Hambantota
Jaffna
Kalmunai
Kalutara
Kandy
Kegalle
Kilinochchi
Kurunegala
Mannar
Matale
Matara
Monaragala
Mullaitivu
Nuwara Eliya
Polonnaruwa
Puttalam
Ratnapura
Trincomalee
Vavuniya


In [15]:
print("POPULATION AREAS")
for area in population_areas:
    print(area)

POPULATION AREAS
Ampara
Anuradhapura
Badulla
Batticaloa
Colombo
Galle
Gampaha
Hambantota
Jaffna
Kalutara
Kandy
Kegalle
Kilinochchi
Kurunegala
Mannar
Matale
Matara
Monaragala
Mullaitivu
Nuwara-eliya
Polonnaruwa
Puttalam
Ratnapura
Trincomalee
Vavuniya


In [16]:
max_length = max(
    len(dengue_areas),
    len(weather_areas),
    len(population_areas)
)
geography_comparison = pd.DataFrame({
    "Dengue": dengue_areas + [None] * (max_length - len(dengue_areas)),
    "Weather": weather_areas + [None] * (max_length - len(weather_areas)),
    "Population": population_areas + [None] * (max_length - len(population_areas))
})

geography_comparison

,Dengue,Weather,Population
0,Ampara,Ampara,Ampara
1,Anuradhapura,Anuradhapura,Anuradhapura
2,Badulla,Badulla,Badulla
3,Batticaloa,Batticaloa,Batticaloa
4,Colombo,Colombo,Colombo
5,Galle,Galle,Galle
6,Gampaha,Gampaha,Gampaha
7,Hambanthota,Hambantota,Hambantota
8,Jaffna,Jaffna,Jaffna
9,Kalmune,Kalmunai,Kalutara


Find exact mismatches: Dengue vs Weather

In [17]:
dengue_set = set(dengue_areas)
weather_set = set(weather_areas)
population_set = set(population_areas)

In [18]:
dengue_not_weather = sorted(dengue_set - weather_set)
weather_not_dengue = sorted(weather_set - dengue_set)
print("Dengue but not Weather:")
print(dengue_not_weather)
print("\nWeather but not Dengue:")
print(weather_not_dengue)

Dengue but not Weather:
['Hambanthota', 'Kalmune', 'NuwaraEliya']

Weather but not Dengue:
['Hambantota', 'Kalmunai', 'Nuwara Eliya']


Compare Dengue vs Population

In [19]:
dengue_not_population = sorted(
    dengue_set - population_set
)
population_not_dengue = sorted(
    population_set - dengue_set
)
print("Dengue but not Population:")
print(dengue_not_population)
print("\nPopulation but not Dengue:")
print(population_not_dengue)

Dengue but not Population:
['Hambanthota', 'Kalmune', 'NuwaraEliya']

Population but not Dengue:
['Hambantota', 'Nuwara-eliya']


Compare Weather vs Population

In [20]:
weather_not_population = sorted(
    weather_set - population_set
)
population_not_weather = sorted(
    population_set - weather_set
)
print("Weather but not Population:")
print(weather_not_population)
print("\nPopulation but not Weather:")
print(population_not_weather)

Weather but not Population:
['Kalmunai', 'Nuwara Eliya']

Population but not Weather:
['Nuwara-eliya']


## 2. Geographic Naming Differences

Comparison of the raw geographic identifiers identified several inconsistencies.

### Likely spelling or formatting differences

- `Hambanthota` in the dengue dataset and `Hambantota` in the weather/population datasets.
- `NuwaraEliya` in dengue, `Nuwara Eliya` in weather, and `Nuwara-eliya` in population.

These appear to be naming-format differences and will be standardized after verification.

### Reporting structure difference

The dengue and weather datasets contain 26 geographic/reporting areas, while the population dataset contains 25 districts.

The dengue identifier `Kalmune` and weather identifier `Kalmunai` do not have a directly matching population identifier.

This difference requires investigation before geographic aggregation or dataset integration.

## 3. Geographic Coverage Over Time

The number of geographic areas represented in each year is examined to determine whether the reporting structure changes during the study period.

In [21]:
dengue_areas_by_year = (
    dengue.groupby("year")["district"]
    .nunique()
)
print("Dengue reporting areas per year:")
print(dengue_areas_by_year)

Dengue reporting areas per year:
year
2010    26
2011    26
2012    26
2013    26
2014    26
2015    26
2016    26
2017    26
2018    26
2019    26
2020    26
2021    26
2022    26
2023    26
2024    26
Name: district, dtype: int64


In [22]:
weather_areas_by_year = (
    weather.groupby("year")["district"]
    .nunique()
)
print("Weather geographic areas per year:")
print(weather_areas_by_year)

Weather geographic areas per year:
year
2010    26
2011    26
2012    26
2013    26
2014    26
2015    26
2016    26
2017    26
2018    26
2019    26
2020    26
2021    26
2022    26
2023    26
2024    26
Name: district, dtype: int64


In [23]:
dengue_years_per_area = (
    dengue.groupby("district")["year"]
    .nunique()
    .sort_index()
)
dengue_years_per_area

district
Ampara          15
Anuradhapura    15
Badulla         15
Batticaloa      15
Colombo         15
Galle           15
Gampaha         15
Hambanthota     15
Jaffna          15
Kalmune         15
Kalutara        15
Kandy           15
Kegalle         15
Kilinochchi     15
Kurunegala      15
Mannar          15
Matale          15
Matara          15
Monaragala      15
Mullaitivu      15
NuwaraEliya     15
Polonnaruwa     15
Puttalam        15
Ratnapura       15
Trincomalee     15
Vavuniya        15
Name: year, dtype: int64

In [24]:
print(
    "Minimum years represented by a dengue area:",
    dengue_years_per_area.min()
)
print(
    "Maximum years represented by a dengue area:",
    dengue_years_per_area.max()
)

Minimum years represented by a dengue area: 15
Maximum years represented by a dengue area: 15


In [25]:
weather_years_per_area = (
    weather.groupby("district")["year"]
    .nunique()
    .sort_index()
)
weather_years_per_area

district
Ampara          15
Anuradhapura    15
Badulla         15
Batticaloa      15
Colombo         15
Galle           15
Gampaha         15
Hambantota      15
Jaffna          15
Kalmunai        15
Kalutara        15
Kandy           15
Kegalle         15
Kilinochchi     15
Kurunegala      15
Mannar          15
Matale          15
Matara          15
Monaragala      15
Mullaitivu      15
Nuwara Eliya    15
Polonnaruwa     15
Puttalam        15
Ratnapura       15
Trincomalee     15
Vavuniya        15
Name: year, dtype: int64

Investigate Kalmune specifically

In [28]:
kalmune_dengue = dengue[
    dengue["district"] == "Kalmune"
].copy()
print("Kalmune dengue records:", len(kalmune_dengue))
print(
    "First year:",
    kalmune_dengue["year"].min()
)
print(
    "Last year:",
    kalmune_dengue["year"].max()
)
kalmune_records_by_year = (
    kalmune_dengue.groupby("year")
    .size()
)

kalmune_records_by_year

Kalmune dengue records: 781
First year: 2010
Last year: 2024


year
2010    52
2011    52
2012    52
2013    52
2014    52
2015    52
2016    53
2017    52
2018    52
2019    52
2020    52
2021    53
2022    52
2023    51
2024    52
dtype: int64

In [29]:
kalmunai_weather = weather[
    weather["district"] == "Kalmunai"
].copy()
print("Kalmunai weather records:", len(kalmunai_weather))
print(
    "First year:",
    kalmunai_weather["year"].min()
)
print(
    "Last year:",
    kalmunai_weather["year"].max()
)
kalmunai_records_by_year = (
    kalmunai_weather.groupby("year")
    .size()
)
kalmunai_records_by_year

Kalmunai weather records: 782
First year: 2010
Last year: 2024


year
2010    52
2011    52
2012    52
2013    52
2014    52
2015    52
2016    53
2017    52
2018    52
2019    52
2020    52
2021    53
2022    52
2023    52
2024    52
dtype: int64

In [30]:
kalmune_counts = (
    kalmune_dengue.groupby("year")
    .size()
    .rename("Kalmune_dengue")
)
kalmunai_counts = (
    kalmunai_weather.groupby("year")
    .size()
    .rename("Kalmunai_weather")
)
kalmune_kalmunai_comparison = pd.concat(
    [kalmune_counts, kalmunai_counts],
    axis=1
)
kalmune_kalmunai_comparison

,Kalmune_dengue,Kalmunai_weather
year,,
2010,52,52
2011,52,52
2012,52,52
2013,52,52
2014,52,52
2015,52,52
2016,53,53
2017,52,52
2018,52,52


In [31]:
print("Population administrative identifiers:")
for district in population_areas:
    print(district)

print(
    "Number of population geographic identifiers:",
    population["district"].nunique()
)

Population administrative identifiers:
Ampara
Anuradhapura
Badulla
Batticaloa
Colombo
Galle
Gampaha
Hambantota
Jaffna
Kalutara
Kandy
Kegalle
Kilinochchi
Kurunegala
Mannar
Matale
Matara
Monaragala
Mullaitivu
Nuwara-eliya
Polonnaruwa
Puttalam
Ratnapura
Trincomalee
Vavuniya
Number of population geographic identifiers: 25


## 4. Geographic Standardization Strategy

The source datasets use two related but different geographic structures.

The dengue dataset follows health reporting areas. The weather dataset contains a corresponding 26 area structure, including Kalmunai. The population dataset contains 25 administrative districts.

Kalmunai is represented separately from Ampara in the health reporting structure, while administratively Kalmunai belongs to Ampara District.

Therefore, two geographic identifiers are retained:

- `reporting_area`: standardized 26-area health/reporting geography
- `administrative_district`: corresponding 25-district administrative geography

This approach preserves the original epidemiological reporting structure while allowing later integration with district level population data.

In [32]:
dengue_std = dengue.copy()
weather_std = weather.copy()
population_std = population.copy()

print("Working copies created.")

Working copies created.


We haveto copy original data because then we work with STD data

In [33]:
dengue_std["source_area"] = dengue_std["district"]
weather_std["source_area"] = weather_std["district"]
population_std["source_area"] = population_std["district"]

In [34]:
dengue_std[
    ["district", "source_area"]
].head()

,district,source_area
0,Colombo,Colombo
1,Gampaha,Gampaha
2,Kalutara,Kalutara
3,Kandy,Kandy
4,Matale,Matale


In [35]:
dengue_name_mapping = {
    "Hambanthota": "Hambantota",
    "NuwaraEliya": "Nuwara Eliya",
    "Kalmune": "Kalmunai"
}

In [36]:
weather_name_mapping = {
    "Hambantota": "Hambantota",
    "Nuwara Eliya": "Nuwara Eliya",
    "Kalmunai": "Kalmunai"
}

In [37]:
population_name_mapping = {
    "Hambantota": "Hambantota",
    "Nuwara-eliya": "Nuwara Eliya"
}

Create standardized reporting area columns

In [38]:
dengue_std["reporting_area"] = (
    dengue_std["district"]
    .replace(dengue_name_mapping)
)

In [43]:
weather_std["reporting_area"] = (
    weather_std["district"]
    .replace(weather_name_mapping)
)

In [44]:
population_std["administrative_district"] = (
    population_std["district"]
    .replace(population_name_mapping)
)

Check the dengue transformation

In [45]:
dengue_changes = dengue_std[
    dengue_std["district"] !=
    dengue_std["reporting_area"]
][
    ["district", "reporting_area"]
].drop_duplicates()

dengue_changes

,district,reporting_area
5,NuwaraEliya,Nuwara Eliya
7,Hambanthota,Hambantota
25,Kalmune,Kalmunai


Check population transformation

In [46]:
population_changes = population_std[
    population_std["district"] !=
    population_std["administrative_district"]
][
    ["district", "administrative_district"]
].drop_duplicates()

population_changes

,district,administrative_district
20,Nuwara-eliya,Nuwara Eliya


Validate dengue and weather standardized names

In [47]:
dengue_standard_areas = set(
    dengue_std["reporting_area"].unique()
)
weather_standard_areas = set(
    weather_std["reporting_area"].unique()
)
print(
    "Dengue standardized reporting areas:",
    len(dengue_standard_areas)
)
print(
    "Weather standardized reporting areas:",
    len(weather_standard_areas)
)
print(
    "Reporting-area sets are identical:",
    dengue_standard_areas == weather_standard_areas
)

Dengue standardized reporting areas: 26
Weather standardized reporting areas: 26
Reporting-area sets are identical: True


Verify there are no remaining name mismatches

In [48]:
print("Dengue but not Weather:")
print(
    sorted(
        dengue_standard_areas -
        weather_standard_areas
    )
)
print("\nWeather but not Dengue:")
print(
    sorted(
        weather_standard_areas -
        dengue_standard_areas
    )
)

Dengue but not Weather:
[]

Weather but not Dengue:
[]


Create administrative-district mapping

In [50]:
reporting_to_admin = {
    area: area
    for area in dengue_standard_areas
}
reporting_to_admin["Kalmunai"] = "Ampara"

reporting_to_admin["Kalmunai"] = "Ampara"

Add administrative district to dengue

In [51]:
dengue_std["administrative_district"] = (
    dengue_std["reporting_area"]
    .map(reporting_to_admin)
)

In [52]:
weather_std["administrative_district"] = (
    weather_std["reporting_area"]
    .map(reporting_to_admin)
)

Inspect Kalmunai mapping

In [53]:
dengue_std[
    dengue_std["reporting_area"] == "Kalmunai"
][
    [
        "source_area",
        "reporting_area",
        "administrative_district"
    ]
].drop_duplicates()

,source_area,reporting_area,administrative_district
25,Kalmune,Kalmunai,Ampara


In [54]:
weather_std[
    weather_std["reporting_area"] == "Kalmunai"
][
    [
        "source_area",
        "reporting_area",
        "administrative_district"
    ]
].drop_duplicates()

,source_area,reporting_area,administrative_district
9,Kalmunai,Kalmunai,Ampara


Validate administrative districts

In [55]:
print(
    "Dengue administrative districts:",
    dengue_std["administrative_district"].nunique()
)
print(
    "Weather administrative districts:",
    weather_std["administrative_district"].nunique()
)
print(
    "Population administrative districts:",
    population_std["administrative_district"].nunique()
)

Dengue administrative districts: 25
Weather administrative districts: 25
Population administrative districts: 25


Compare administrative names across all datasets

In [57]:
dengue_admin_set = set(
    dengue_std["administrative_district"].unique()
)
weather_admin_set = set(
    weather_std["administrative_district"].unique()
)
population_admin_set = set(
    population_std["administrative_district"].unique()
)
print(
    "Dengue = Weather:",
    dengue_admin_set == weather_admin_set
)
print(
    "Dengue = Population:",
    dengue_admin_set == population_admin_set
)
print(
    "Weather = Population:",
    weather_admin_set == population_admin_set
)

Dengue = Weather: True
Dengue = Population: True
Weather = Population: True


In [58]:
dengue_std[
    dengue_std["administrative_district"] == "Ampara"
][
    ["reporting_area", "administrative_district"]
].drop_duplicates()

,reporting_area,administrative_district
15,Ampara,Ampara
25,Kalmunai,Ampara


Check unmapped values

In [ ]:
print(
    "Unmapped dengue administrative districts:",
    dengue_std["administrative_district"].isnull().sum()
)
print(
    "Unmapped weather administrative districts:",
    weather_std["administrative_district"].isnull().sum()
)
print(
    "Unmapped population administrative districts:",
    population_std["administrative_district"].isnull().sum()
)

Unmapped dengue administrative districts: 0
Unmapped weather administrative districts: 0
Unmapped population administrative districts: 0


## 5. Canonical Geographic Mapping

A canonical mapping table is created to preserve the relationship between the 26 health/reporting areas and the 25 administrative districts.

Most reporting areas correspond directly to an administrative district. Kalmunai is retained as a separate reporting area but mapped to Ampara at the administrative district level.

In [60]:
geography_mapping = (
    dengue_std[
        ["reporting_area", "administrative_district"]
    ]
    .drop_duplicates()
    .sort_values("reporting_area")
    .reset_index(drop=True)
)
geography_mapping

,reporting_area,administrative_district
0,Ampara,Ampara
1,Anuradhapura,Anuradhapura
2,Badulla,Badulla
3,Batticaloa,Batticaloa
4,Colombo,Colombo
5,Galle,Galle
6,Gampaha,Gampaha
7,Hambantota,Hambantota
8,Jaffna,Jaffna
9,Kalmunai,Ampara


In [61]:
print("Mapping rows:", len(geography_mapping))
print(
    "Unique reporting areas:",
    geography_mapping["reporting_area"].nunique()
)
print(
    "Unique administrative districts:",
    geography_mapping["administrative_district"].nunique()
)

Mapping rows: 26
Unique reporting areas: 26
Unique administrative districts: 25


Final reporting area key validation

In [63]:
dengue_duplicate_keys_after = dengue_std.duplicated(
    subset=["year", "week", "reporting_area"]
).sum()
weather_duplicate_keys_after = weather_std.duplicated(
    subset=["year", "week", "reporting_area"]
).sum()
print(
    "Dengue duplicate year-week-reporting-area keys:",
    dengue_duplicate_keys_after
)
print(
    "Weather duplicate year-week-reporting-area keys:",
    weather_duplicate_keys_after
)

Dengue duplicate year-week-reporting-area keys: 0
Weather duplicate year-week-reporting-area keys: 0


In [65]:
population_duplicate_keys = population_std.duplicated(
    subset=["year", "administrative_district"]
).sum()
print(
    "Duplicate population year-district keys:",
    population_duplicate_keys
)

Duplicate population year-district keys: 200


### Population Duplicate Records

The population dataset contains duplicate district year observations identified during the initial data inventory.

These records are intentionally retained in this notebook because the purpose of the current stage is geographic standardization rather than population data cleaning.

Population duplicate handling will be performed separately during population data preparation.

In [66]:
geography_validation = pd.DataFrame({
    "Validation Check": [
        "Dengue standardized reporting areas",
        "Weather standardized reporting areas",
        "Reporting-area sets identical",
        "Dengue administrative districts",
        "Weather administrative districts",
        "Population administrative districts",
        "Dengue and weather admin sets identical",
        "Dengue and population admin sets identical",
        "Weather and population admin sets identical",
        "Unmapped dengue administrative districts",
        "Unmapped weather administrative districts",
        "Unmapped population administrative districts",
        "Dengue duplicate reporting-area keys after standardization",
        "Weather duplicate reporting-area keys after standardization"
    ],
    "Result": [
        dengue_std["reporting_area"].nunique(),
        weather_std["reporting_area"].nunique(),
        dengue_standard_areas == weather_standard_areas,
        dengue_std["administrative_district"].nunique(),
        weather_std["administrative_district"].nunique(),
        population_std["administrative_district"].nunique(),
        dengue_admin_set == weather_admin_set,
        dengue_admin_set == population_admin_set,
        weather_admin_set == population_admin_set,
        dengue_std["administrative_district"].isna().sum(),
        weather_std["administrative_district"].isna().sum(),
        population_std["administrative_district"].isna().sum(),
        dengue_duplicate_keys_after,
        weather_duplicate_keys_after
    ]
})

geography_validation

,Validation Check,Result
0,Dengue standardized reporting areas,26
1,Weather standardized reporting areas,26
2,Reporting-area sets identical,True
3,Dengue administrative districts,25
4,Weather administrative districts,25
5,Population administrative districts,25
6,Dengue and weather admin sets identical,True
7,Dengue and population admin sets identical,True
8,Weather and population admin sets identical,True
9,Unmapped dengue administrative districts,0


In [67]:
row_count_validation = pd.DataFrame({
    "Dataset": [
        "Dengue",
        "Weather",
        "Population"
    ],
    "Raw Rows": [
        len(dengue),
        len(weather),
        len(population)
    ],
    "Standardized Rows": [
        len(dengue_std),
        len(weather_std),
        len(population_std)
    ]
})

row_count_validation["Rows Unchanged"] = (
    row_count_validation["Raw Rows"] ==
    row_count_validation["Standardized Rows"]
)

row_count_validation

,Dataset,Raw Rows,Standardized Rows,Rows Unchanged
0,Dengue,20306,20306,True
1,Weather,20332,20332,True
2,Population,475,475,True


Create the interim directory safely

In [68]:
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)
print("Interim directory:", INTERIM_DIR)

Interim directory: d:\Dengue_Project\data\interim


In [69]:
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)
print("Interim directory:", INTERIM_DIR)

Interim directory: d:\Dengue_Project\data\interim


In [71]:
dengue_output = (
    INTERIM_DIR /
    "dengue_weekly_geography_standardized.csv"
)
weather_output = (
    INTERIM_DIR /
    "weather_weekly_geography_standardized.csv"
)
population_output = (
    INTERIM_DIR /
    "population_geography_standardized.csv"
)
mapping_output = (
    INTERIM_DIR /
    "geography_mapping.csv"
)

In [72]:
print(dengue_output)
print(weather_output)
print(population_output)
print(mapping_output)

d:\Dengue_Project\data\interim\dengue_weekly_geography_standardized.csv
d:\Dengue_Project\data\interim\weather_weekly_geography_standardized.csv
d:\Dengue_Project\data\interim\population_geography_standardized.csv
d:\Dengue_Project\data\interim\geography_mapping.csv


In [73]:
dengue_std.to_csv(
    dengue_output,
    index=False
)
weather_std.to_csv(
    weather_output,
    index=False
)
population_std.to_csv(
    population_output,
    index=False
)
geography_mapping.to_csv(
    mapping_output,
    index=False
)
print("Standardized geography files saved successfully.")

Standardized geography files saved successfully.


In [74]:
outputs = [
    dengue_output,
    weather_output,
    population_output,
    mapping_output
]
for file in outputs:
    print(
        file.name,
        "->",
        file.exists()
    )

dengue_weekly_geography_standardized.csv -> True
weather_weekly_geography_standardized.csv -> True
population_geography_standardized.csv -> True
geography_mapping.csv -> True


In [75]:
dengue_check = pd.read_csv(dengue_output)
weather_check = pd.read_csv(weather_output)
population_check = pd.read_csv(population_output)
mapping_check = pd.read_csv(mapping_output)
print("Reloaded dengue:", dengue_check.shape)
print("Reloaded weather:", weather_check.shape)
print("Reloaded population:", population_check.shape)
print("Reloaded mapping:", mapping_check.shape)

Reloaded dengue: (20306, 9)
Reloaded weather: (20332, 22)
Reloaded population: (475, 5)
Reloaded mapping: (26, 2)


In [76]:
print(
    "Reloaded dengue reporting areas:",
    dengue_check["reporting_area"].nunique()
)
print(
    "Reloaded weather reporting areas:",
    weather_check["reporting_area"].nunique()
)
print(
    "Reloaded dengue administrative districts:",
    dengue_check["administrative_district"].nunique()
)
print(
    "Reloaded weather administrative districts:",
    weather_check["administrative_district"].nunique()
)

print(
    "Reloaded population administrative districts:",
    population_check["administrative_district"].nunique()
)

Reloaded dengue reporting areas: 26
Reloaded weather reporting areas: 26
Reloaded dengue administrative districts: 25
Reloaded weather administrative districts: 25
Reloaded population administrative districts: 25


## Key Findings

1. The dengue and weather datasets consistently contain 26 geographic/reporting areas across the study period.

2. The population dataset contains 25 administrative districts.

3. Geographic naming inconsistencies were identified for Hambantota, Nuwara Eliya, and Kalmunai across the source datasets.

4. The dengue source name `Hambanthota` was standardized to `Hambantota`.

5. The dengue source name `NuwaraEliya`, weather name `Nuwara Eliya`, and population name `Nuwara-eliya` were standardized to `Nuwara Eliya`.

6. The dengue reporting area name `Kalmune` was standardized to `Kalmunai` to align with the corresponding weather reporting-area identifier.

7. The 26-area health/reporting geography was preserved rather than forcing the data immediately into a 25 area structure.

8. A separate `administrative_district` identifier was introduced. Kalmunai is retained as a reporting area while being associated with Ampara at the administrative-district level.

9. After standardization, the dengue and weather datasets contain identical sets of 26 reporting-area identifiers.

10. At the administrative-district level, dengue, weather, and population data contain the same 25 standardized district identifiers.

11. No observations were removed during geographic standardization.

12. Population duplicate records were intentionally retained because duplicate handling belongs to the later population data preparation stage.

## Conclusion

Geographic identifiers across the dengue, weather and population datasets were successfully standardized while preserving the distinction between health reporting areas and administrative districts.

The dengue and weather datasets retain a 26 area reporting structure, while a corresponding 25 district administrative identifier was introduced to support later integration with population data. Kalmunai is therefore preserved as a distinct reporting area rather than being prematurely combined with Ampara.

Spelling and formatting inconsistencies involving Hambantota, Nuwara Eliya, and Kalmunai were standardized using explicit mappings. After standardization, dengue and weather reporting area identifiers are consistent and the administrative district identifiers are consistent across all three datasets.

No raw files were modified, no observations were removed, and no geographic aggregation was performed in this notebook. Population duplicate handling and any required geographic aggregation will be performed in their appropriate preprocessing stages.

The standardized datasets and geographic mapping table were saved to the interim data directory for use in subsequent preprocessing notebooks.